# RegistrySpec loading demonstration

This is a new, standalone demonstration of the Phase 4A registry specification loader. It uses the target-format NSCLC registry, does not open a database, and does not migrate or modify the older notebooks.

## Locate the repository and import the target APIs

`RegistrySpec.from_path()` takes only a path. Condition names and CUIs remain declarations until a vocabulary gateway is supplied explicitly.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys
import tempfile


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scoop.protocols.spec import RegistrySpec
from scoop.vocabulary.gateway import FakeVocabularyGateway

REGISTRY_PATH = ROOT / "scoop" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"
REGISTRY_PATH

## Load the registry without a session

In [ ]:
registry = RegistrySpec.from_path(REGISTRY_PATH)

summary = {
    "label": registry.label,
    "version": registry.version,
    "default_condition_names": registry.default_conditions.names,
    "source_phase_count": len(registry.source_phases),
    "phase_choice_count": len(registry.union_phases),
    "protocol_count": len(registry.protocols),
}
pprint(summary)

## Inspect preserved declarations and derived phase categories

The loader preserves condition names as `ConditionRef.names`; it does not turn them into CUIs during parsing. `delivered_categories()` is derived from the declared component rules.

In [ ]:
phase_rows = []
for phase_id, phase in registry.source_phases.items():
    phase_rows.append(
        {
            "phase_id": phase_id,
            "type": phase.type,
            "conditions": phase.conditions.names or phase.conditions.cuis,
            "delivered_categories": [category.value for category in sorted(phase.delivered_categories() or [])],
            "component_lists": phase.component_requirements.component_lists,
        }
    )

pprint(phase_rows[:8])
assert registry.default_conditions.names == ["Lung cancer", "Non-small cell lung cancer"]

## Resolve conditions only when requested

This fake gateway keeps the example database-free while showing the intended resolution boundary.

In [ ]:
vocabulary = FakeVocabularyGateway(
    condition_names={
        "Lung cancer": 1001,
        "Non-small cell lung cancer": 1002,
    }
)

resolved_conditions = registry.default_conditions.resolve(vocabulary)
print("declared:", registry.default_conditions.names)
print("resolved:", resolved_conditions)
assert resolved_conditions == (1001, 1002)

## Invalid references fail during loading

A protocol that names an undeclared phase is rejected before any candidate query or clinical resolution takes place.

In [ ]:
import yaml

broken_registry = {
    "fixture": {
        "version": 1,
        "conditions": [1001],
        "source_phases": {"known": {}},
        "protocols": {"broken": {"phases": ["missing"]}},
    }
}

with tempfile.TemporaryDirectory() as directory:
    broken_path = Path(directory) / "broken_registry.yaml"
    broken_path.write_text(yaml.safe_dump(broken_registry), encoding="utf-8")
    try:
        RegistrySpec.from_path(broken_path)
    except Exception as error:
        print(type(error).__name__ + ":", error)
        assert "broken" in str(error)
    else:
        raise AssertionError("The dangling phase reference was accepted.")

## Round-trip the typed specification

The canonical YAML representation can be written and loaded again without changing the typed `RegistrySpec`.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    round_trip_path = Path(directory) / "round_trip.yaml"
    round_trip_path.write_text(registry.to_yaml(), encoding="utf-8")
    round_tripped = RegistrySpec.from_path(round_trip_path)

assert round_tripped == registry
print("Round-trip preserved the typed registry.")